In [ ]:
import zipfile
with zipfile.ZipFile("/your_path", 'r') as z:
    print(z.namelist())

['metadata.json', 'config.json', 'model.weights.h5']


In [ ]:
import zipfile
import h5py
import io

with zipfile.ZipFile("/your_path", 'r') as z:
    with z.open('model.weights.h5') as f:
        data = io.BytesIO(f.read())

with h5py.File(data, 'r') as f:
    def print_keys(name):
        print(name)
    f.visititems(lambda name, obj: print(name))

layers
layers/dense
layers/dense/vars
layers/dense/vars/0
layers/dense/vars/1
layers/dense_1
layers/dense_1/vars
layers/dense_1/vars/0
layers/dense_1/vars/1
layers/dense_2
layers/dense_2/vars
layers/dense_2/vars/0
layers/dense_2/vars/1
layers/dropout
layers/dropout/vars
layers/dropout_1
layers/dropout_1/vars
layers/gated_residual_network
layers/gated_residual_network/dense_elu
layers/gated_residual_network/dense_elu/vars
layers/gated_residual_network/dense_elu/vars/0
layers/gated_residual_network/dense_elu/vars/1
layers/gated_residual_network/dense_gate
layers/gated_residual_network/dense_gate/vars
layers/gated_residual_network/dense_gate/vars/0
layers/gated_residual_network/dense_gate/vars/1
layers/gated_residual_network/dense_linear
layers/gated_residual_network/dense_linear/vars
layers/gated_residual_network/dense_linear/vars/0
layers/gated_residual_network/dense_linear/vars/1
layers/gated_residual_network/dropout
layers/gated_residual_network/dropout/vars
layers/gated_residual_netw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
import json

with zipfile.ZipFile("/your_path", 'r') as z:
    with z.open('metadata.json') as f:
        print(json.load(f))

Mounted at /content/drive
{'keras_version': '3.13.2', 'date_saved': '2026-03-30@08:49:35'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Must be FIRST line before any TF import

import tensorflow as tf
import numpy as np
import keras.src.saving.saving_lib as saving_lib

class TemporalSumLayer(tf.keras.layers.Layer):
    def call(self, x): return tf.reduce_sum(x, axis=1)

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, **kwargs): super().__init__(**kwargs)
    def build(self, input_shape):
        time_steps, d_model = input_shape[1], input_shape[2]
        angles = np.arange(time_steps)[:, np.newaxis] / np.power(10000, (2*(np.arange(d_model)[np.newaxis, :]//2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.constant(angles[np.newaxis], dtype=tf.float32)
        super().build(input_shape)
    def call(self, x): return x + self.pe

class GatedResidualNetwork(tf.keras.layers.Layer):
    def __init__(self, units=128, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dense_elu = tf.keras.layers.Dense(units, activation='elu')
        self.dense_linear = tf.keras.layers.Dense(units)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.dense_gate = tf.keras.layers.Dense(units, activation='sigmoid')
        self.layer_norm = tf.keras.layers.LayerNormalization()
        self.dense_proj = None
    def build(self, input_shape):
        if input_shape[-1] != self.units: self.dense_proj = tf.keras.layers.Dense(self.units)
        super().build(input_shape)
    def call(self, x, training=False):
        residual = x if self.dense_proj is None else self.dense_proj(x)
        h = self.dense_linear(self.dense_elu(x))
        h = self.dropout(h, training=training) * self.dense_gate(h)
        return self.layer_norm(h + residual)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model=128, n_heads=4, ff_dim=128, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.n_heads = n_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = tf.keras.layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model//n_heads, dropout=dropout_rate)
        self.drop1, self.drop2, self.drop3 = [tf.keras.layers.Dropout(dropout_rate) for _ in range(3)]
        self.norm1, self.norm2 = [tf.keras.layers.LayerNormalization(epsilon=1e-6) for _ in range(2)]
        self.ff1 = tf.keras.layers.Dense(ff_dim, activation='gelu')
        self.ff2 = tf.keras.layers.Dense(d_model)
    def build(self, input_shape):
        dummy = tf.zeros([1] + list(input_shape[1:]))
        self.attn(dummy, dummy)
        self.ff1(dummy)
        self.ff2(dummy)
        super().build(input_shape)
    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'n_heads': self.n_heads, 'ff_dim': self.ff_dim, 'dropout_rate': self.dropout_rate})
        return config
    def call(self, x, training=False):
        x = self.norm1(x + self.drop1(self.attn(x, x, training=training), training=training))
        return self.norm2(x + self.drop3(self.ff2(self.drop2(self.ff1(x), training=training)), training=training))

class VariableSelectionNetwork(tf.keras.layers.Layer):
    def __init__(self, n_features=13, d_model=128, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.n_features = n_features
        self.feature_projs = [tf.keras.layers.Dense(d_model) for _ in range(n_features)]
        self.feature_grns = [GatedResidualNetwork(d_model, dropout_rate) for _ in range(n_features)]
        self.selection_proj = tf.keras.layers.Dense(d_model)
        self.selection_grn = GatedResidualNetwork(d_model, dropout_rate)
        self.selection_gate = tf.keras.layers.Dense(n_features, activation='softmax')
    def call(self, x, training=False):
        var_outs = [self.feature_grns[i](self.feature_projs[i](x[:, :, i:i+1]), training=training) for i in range(self.n_features)]
        weights = tf.expand_dims(self.selection_gate(self.selection_grn(self.selection_proj(x), training=training)), axis=-1)
        return tf.reduce_sum(tf.stack(var_outs, axis=2) * weights, axis=2)

CUSTOM_OBJECTS = {
    'TemporalSumLayer': TemporalSumLayer,
    'PositionalEncoding': PositionalEncoding,
    'GatedResidualNetwork': GatedResidualNetwork,
    'TransformerBlock': TransformerBlock,
    'VariableSelectionNetwork': VariableSelectionNetwork,
}

# Patch Keras to warn instead of crash
original_raise = saving_lib._raise_loading_failure

saving_lib._raise_loading_failure = lambda error_msgs: None

cnn_s1 = tf.keras.models.load_model("/content/drive/MyDrive/GNSS_csir/stage1_classifier.keras", custom_objects=CUSTOM_OBJECTS, compile=False)
tft_s1 = tf.keras.models.load_model("/your_path", custom_objects=CUSTOM_OBJECTS, compile=False)
cnn_s2 = tf.keras.models.load_model("/content/drive/MyDrive/GNSS_csir/stage2_regressor.keras", custom_objects=CUSTOM_OBJECTS, compile=False)

cnn_s1.export("/content/drive/MyDrive/GNSS_csir/cnn_s1_saved")
tft_s1.export("/content/drive/MyDrive/GNSS_csir/tft_s1_saved")
cnn_s2.export("/content/drive/MyDrive/GNSS_csir/cnn_s2_saved")

print("Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'variable_selection_network', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Saved artifact at '/content/drive/MyDrive/GNSS_csir/cnn_s1_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 13), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138778811249104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811250448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811251408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811250640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811251216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811249872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811252176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811251024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811252560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138778811252368: TensorSpec(shape=(), dtype=tf.resource, name